# DistilBERT fine-tuning

The optional throughput pilot has been skipped. Token coverage fixed `max_length=256`; training starts conservatively with physical batch size 8 and four gradient-accumulation steps for an effective batch of 32. This notebook now performs the required two-epoch fine-tuning run and writes resumable checkpoints plus the final model bundle directly to private Google Drive.

Before running all cells:

1. In Colab, choose **Runtime > Change runtime type > GPU**.
2. Commit and push the repository version that contains this notebook.
3. Upload the matching `training_dataset.parquet` to `MyDrive/AIDI_artefact/private/`.

Training reads only `train` and `validation_model_selection`; policy-calibration and test rows remain untouched. The notebook never commits or pushes changes. Package pins are installed into a project-only dependency directory, so no virtual environment or runtime restart is required. Cells are safe to run again and training resumes from the latest compatible epoch checkpoint.

In [ ]:
import shutil
import subprocess
import sys
import os
from pathlib import Path

if shutil.which('nvidia-smi') is None:
    raise RuntimeError(
        'No NVIDIA GPU runtime was found. In Colab choose Runtime > Change runtime type > GPU, then run again.'
    )
gpu_check = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True, check=False,
)
if gpu_check.returncode != 0:
    raise RuntimeError('The Colab GPU is not ready:\n' + gpu_check.stderr.strip())
print('GPU:', gpu_check.stdout.strip())

In [ ]:
try:
    from google.colab import drive
except ImportError as error:
    raise RuntimeError('This notebook must be run in Google Colab.') from error

drive.mount('/content/drive', force_remount=False)

REPO_URL = 'https://github.com/MurrayMint7/AIDI_artefact.git'
REPO_ROOT = Path('/content/AIDI_artefact')
PRIVATE_ROOT = Path('/content/drive/MyDrive/AIDI_artefact/private')
DATASET_PATH = PRIVATE_ROOT / 'training_dataset.parquet'
MODEL_CACHE = PRIVATE_ROOT / 'huggingface-cache'
MODEL_RUN_DIR = PRIVATE_ROOT / 'distilbert-run'
DEPS_ROOT = Path('/content/aidi-distilbert-deps')

if not DATASET_PATH.is_file():
    raise FileNotFoundError(
        f'Dataset not found at {DATASET_PATH}. Upload the governed Parquet file there, then rerun this cell.'
    )
print('Dataset:', DATASET_PATH)

In [ ]:
if REPO_ROOT.exists():
    if not (REPO_ROOT / '.git').is_dir():
        raise RuntimeError(f'{REPO_ROOT} exists but is not a Git clone. Rename it or delete the Colab runtime.')
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
    print(f'Updated existing clone at {REPO_ROOT}')
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_ROOT)], check=True)

required_files = [
    REPO_ROOT / 'requirements-colab.txt',
    REPO_ROOT / 'requirements-transformer.txt',
    REPO_ROOT / 'config/distilbert.yaml',
    REPO_ROOT / 'artifacts/metrics/distilbert_token_length_summary.json',
    REPO_ROOT / 'artifacts/metrics/distilbert_training_decision.json',
]
missing = [str(path.relative_to(REPO_ROOT)) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError('The clone is missing required file(s): ' + ', '.join(missing))
commit = subprocess.run(
    ['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'],
    capture_output=True, text=True, check=True,
).stdout.strip()
print('Repository commit:', commit)

In [ ]:
# Keep project pins separate without replacing Colab's matched Torch/CUDA stack.
if DEPS_ROOT.exists():
    shutil.rmtree(DEPS_ROOT)
DEPS_ROOT.mkdir(parents=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-q',
     '--no-deps', '--target', str(DEPS_ROOT),
     '-r', str(REPO_ROOT / 'requirements-colab.txt')],
    check=True,
)
RUN_PYTHON = sys.executable
RUN_ENV = os.environ.copy()
python_paths = [str(REPO_ROOT / 'src'), str(DEPS_ROOT)]
if RUN_ENV.get('PYTHONPATH'):
    python_paths.append(RUN_ENV['PYTHONPATH'])
RUN_ENV['PYTHONPATH'] = os.pathsep.join(python_paths)
RUN_ENV['PYTHONNOUSERSITE'] = '1'
print('Project dependency directory ready:', DEPS_ROOT)

In [ ]:
# Validate imports, CUDA visibility, Parquet schema, and the frozen dataset/training decision before downloading the model.
preflight_code = r'''
import hashlib
import importlib.metadata
import json
import sys
from pathlib import Path

import pyarrow.parquet as pq
import torch
from transformers import Trainer

dataset_path = Path(sys.argv[1])
decision_path = Path(sys.argv[2])
deps_root = Path(sys.argv[3]).resolve()
torch_path = Path(torch.__file__).resolve()
if torch_path.is_relative_to(deps_root):
    raise RuntimeError(
        f'Project dependencies replaced Colab Torch at {torch_path}. '
        'Rerun the dependency setup cell to rebuild the clean dependency directory.'
    )
required_columns = {'text', 'sentiment_label', 'source_category', 'split'}
columns = set(pq.ParquetFile(dataset_path).schema_arrow.names)
missing_columns = sorted(required_columns - columns)
if missing_columns:
    raise RuntimeError(f'Dataset is missing required columns: {missing_columns}')
decision = json.loads(decision_path.read_text(encoding='utf-8'))
dataset_hash = hashlib.sha256(dataset_path.read_bytes()).hexdigest()
expected_hash = decision.get('dataset_sha256')
if dataset_hash != expected_hash:
    raise RuntimeError(
        'The Drive dataset does not match the frozen training decision in this repository. '
        'Upload the exact training_dataset.parquet used for the token-length analysis.'
    )
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch cannot see CUDA. Select a GPU runtime and rerun from the first cell.')
for package in ['torch', 'transformers', 'pandas', 'pyarrow', 'fsspec']:
    print(package, importlib.metadata.version(package))
print('Torch loaded from:', torch_path)
print('CUDA device:', torch.cuda.get_device_name(0))
print('Dataset and training decision match:', dataset_hash)
'''
subprocess.run(
    [RUN_PYTHON, '-c', preflight_code, str(DATASET_PATH),
     str(REPO_ROOT / 'artifacts/metrics/distilbert_training_decision.json'),
     str(DEPS_ROOT)],
    cwd=REPO_ROOT, env=RUN_ENV, check=True,
)

In [ ]:
command = [
    RUN_PYTHON, '-m', 'amazon_sentiment', 'train-transformer',
    '--config', 'config/distilbert.yaml',
    '--dataset', str(DATASET_PATH),
    '--decision', 'artifacts/metrics/distilbert_training_decision.json',
    '--output-dir', str(MODEL_RUN_DIR),
    '--cache-dir', str(MODEL_CACHE),
]
process = subprocess.Popen(
    command, cwd=REPO_ROOT, env=RUN_ENV,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
last_lines = []
for line in process.stdout:
    print(line, end='')
    last_lines.append(line)
    last_lines = last_lines[-100:]
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(
        'DistilBERT training failed. Last output:\n' + ''.join(last_lines)
    )
print('DistilBERT fine-tuning completed.')

In [ ]:
import json

required_outputs = [
    MODEL_RUN_DIR / 'model' / 'config.json',
    MODEL_RUN_DIR / 'model' / 'trainer_state.json',
    MODEL_RUN_DIR / 'validation_model_selection_outputs.npz',
    MODEL_RUN_DIR / 'training_summary.json',
    MODEL_RUN_DIR / 'environment.json',
]
missing_outputs = [str(path) for path in required_outputs if not path.exists()]
if missing_outputs:
    raise FileNotFoundError('Training did not create required output(s): ' + ', '.join(missing_outputs))
summary = json.loads((MODEL_RUN_DIR / 'training_summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, indent=2))
print('Private model bundle:', MODEL_RUN_DIR)